In [1]:
import os, sys, pandas as pd
from pathlib import Path

ROOT = Path('/home/coders/Escritorio/emausoft-analytics')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f'Directorio: {os.getcwd()}')

# Carga de datos

df_ventas    = pd.read_csv(ROOT / 'data/interim/ventas_con_producto.csv')
df_clientes  = pd.read_csv(ROOT / 'data/interim/clientes.csv')
df_productos = pd.read_csv(ROOT / 'data/interim/productos.csv')
df_paises    = pd.read_csv(ROOT / 'data/interim/paises.csv')
print(f'Ventas:    {df_ventas.shape}')
print(f'Clientes:  {df_clientes.shape}')
print(f'Productos: {df_productos.shape}')
print(f'Paises:    {df_paises.shape}')

# Dimensiones
df_ventas['ORDERDATE'] = pd.to_datetime(df_ventas['ORDERDATE'], format='mixed')
fechas = pd.DataFrame({'fecha': pd.date_range(
    df_ventas['ORDERDATE'].min(),
    df_ventas['ORDERDATE'].max())})
fechas['fecha_id']   = fechas['fecha'].dt.strftime('%Y%m%d').astype(int)
fechas['ano']        = fechas['fecha'].dt.year
fechas['mes']        = fechas['fecha'].dt.month
fechas['trimestre']  = fechas['fecha'].dt.quarter
fechas['dia']        = fechas['fecha'].dt.day
fechas['dia_semana'] = fechas['fecha'].dt.day_name()
df_ventas['fecha_id'] = df_ventas['ORDERDATE'].dt.strftime('%Y%m%d').astype(int)
print(f'\ndim_fecha: {fechas.shape}')

# Geografia

MAPEO = {'USA': 'United States', 'UK': 'United Kingdom'}
df_ventas['pais_norm'] = df_ventas['COUNTRY'].replace(MAPEO)
dim_geo = (df_paises[['pais','iso2','iso3','region','subregion','lat','lng']]
           .drop_duplicates().reset_index(drop=True))
dim_geo.insert(0, 'geografia_id', range(1, len(dim_geo)+1))
df_ventas = df_ventas.merge(
    dim_geo[['geografia_id','pais']],
    left_on='pais_norm', right_on='pais', how='left')

sin_match = df_ventas[df_ventas['geografia_id'].isnull()]['COUNTRY'].unique()
if len(sin_match) > 0:
    print(f'Paises sin match: {sin_match}')
else:
    print('Todos los paises encontraron match')

# Fact table

cols_fact = ['ORDERNUMBER','producto_id','cliente_id',
             'fecha_id','geografia_id',
             'QUANTITYORDERED','PRICEEACH','SALES','STATUS']
fact = df_ventas[cols_fact].copy()
fact.columns = ['order_number','producto_id','cliente_id',
                'fecha_id','geografia_id',
                'quantity_ordered','price_each','sales','status']
print(f'\nfact_ventas: {fact.shape}')
print(f'Nulos cliente_id:   {fact["cliente_id"].isnull().sum()}')
print(f'Nulos producto_id:  {fact["producto_id"].isnull().sum()}')
print(f'Nulos fecha_id:     {fact["fecha_id"].isnull().sum()}')
print(f'Nulos geografia_id: {fact["geografia_id"].isnull().sum()}')

# Guardar archivos

proc = ROOT / 'data/processed'
proc.mkdir(parents=True, exist_ok=True)

fact.to_csv(proc / 'fact_ventas.csv',      index=False)
df_clientes.to_csv(proc / 'dim_clientes.csv',   index=False)
df_productos.to_csv(proc / 'dim_productos.csv',  index=False)
dim_geo.to_csv(proc / 'dim_geografia.csv',  index=False)
fechas.to_csv(proc / 'dim_fecha.csv',       index=False)

print('\nArchivos guardados en data/processed/:')
print('  fact_ventas.csv')
print('  dim_clientes.csv')
print('  dim_productos.csv')
print('  dim_geografia.csv')
print('  dim_fecha.csv')

Directorio: /home/coders/Escritorio/emausoft-analytics
Ventas:    (2823, 34)
Clientes:  (100, 6)
Productos: (109, 4)
Paises:    (250, 7)

dim_fecha: (877, 7)
Todos los paises encontraron match

fact_ventas: (2823, 9)
Nulos cliente_id:   0
Nulos producto_id:  0
Nulos fecha_id:     0
Nulos geografia_id: 0

Archivos guardados en data/processed/:
  fact_ventas.csv
  dim_clientes.csv
  dim_productos.csv
  dim_geografia.csv
  dim_fecha.csv
